In [60]:
import numpy as np

data = [
    [12.0, 1.5, 1, 'Wine'],
    [5.0, 2.0, 0, 'Beer'],
    [40.0, 0.0, 1, 'Whiskey'],
    [13.5, 1.2, 1, 'Wine'],
    [4.5, 1.8, 0, 'Beer'],
    [38.0, 0.1, 1, 'Whiskey'],
    [11.5, 1.7, 1, 'Wine'],
    [5.5, 2.3, 0, 'Beer']
]

# Label encoding
labels = {'Wine': 0, 'Beer': 1, 'Whiskey': 2}

# name maps for pretty printing
reverse_labels = {
    0: 'Wine',
    1: 'Beer',
    2: 'Whiskey'
}
feature_names = {
    0: "Alcohol",
    1: "Sugar",
    2: "Color"
}

X = np.array([row[:3] for row in data], dtype=float)
y = np.array([labels[row[3]] for row in data])

In [61]:
def gini(y):
  classes, counts = np.unique(y, return_counts=True)
  prob = counts / len(y)
  return 1 - np.sum(prob ** 2)

In [62]:
def best_split(X, y):
    best_gini = float('inf')
    best_feature = None
    best_threshold = None

    n_samples, n_features = X.shape

    for feature in range(n_features):
        thresholds = np.unique(X[:, feature])
        # better thresholds = midpoint values of unique vals
        if len(thresholds) > 1:
            thresholds = (thresholds[:-1] + thresholds[1:]) / 2

        for threshold in thresholds:
            left_idx = X[:, feature] <= threshold
            right_idx = X[:, feature] > threshold

            if len(y[left_idx]) == 0 or len(y[right_idx]) == 0:
                continue

            gini_left = gini(y[left_idx])
            gini_right = gini(y[right_idx])

            # group sizes may be unequal, so considering weighted gini for
            weighted_gini = (
                len(y[left_idx]) * gini_left +
                len(y[right_idx]) * gini_right
            ) / len(y)

            if weighted_gini < best_gini:
                best_gini = weighted_gini
                best_feature = feature
                best_threshold = threshold

    return best_feature, best_threshold

In [63]:
class Node:
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, value=None):
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left #left if True, right if False
        self.right = right
        self.value = value # predicted class (if leaf node)

In [68]:
class DecisionTree:
    def __init__(self, max_depth=3, min_samples=2):
        self.max_depth = max_depth
        self.min_samples = min_samples

    def majority_class(self, y):
        return np.bincount(y).argmax()

    def build_tree(self, X, y, depth):
        # Stopping/Leaf node conditions
        if (
            depth >= self.max_depth or
            len(np.unique(y)) == 1 or
            len(y) < self.min_samples
        ):
            leaf_value = self.majority_class(y)
            return Node(value=leaf_value)

        feature, threshold = best_split(X, y)

        if feature is None:
            return Node(value=self.majority_class(y))

        left_idx = X[:, feature] <= threshold
        right_idx = X[:, feature] > threshold

        left = self.build_tree(X[left_idx], y[left_idx], depth + 1)
        right = self.build_tree(X[right_idx], y[right_idx], depth + 1)

        return Node(feature, threshold, left, right)

    def fit(self, X, y):
        self.root = self.build_tree(X, y, depth=0)

    def predict_one(self, x, node):
        # if leaf, return class
        if node.value is not None:
            return node.value

        if x[node.feature_index] <= node.threshold:
            return self.predict_one(x, node.left)
        else:
            return self.predict_one(x, node.right)

    def predict(self, X):
        return np.array([self.predict_one(x, self.root) for x in X])

    def print_tree(self, node=None, depth=0):
        # start from root by default
        if node is None:
            node = self.root

        indent = "  " * depth

        # if leaf node
        if node.value is not None:
            print(indent + f"{reverse_labels[node.value]} ({node.value})")
        else:
            print(indent + f"if {feature_names[node.feature_index]} (ft {node.feature_index}) <= {node.threshold}:")
            self.print_tree(node.left, depth + 1)
            print(indent + "else:")
            self.print_tree(node.right, depth + 1)

In [69]:
test_data = np.array([
    [6.0, 2.1, 0],   # Expected: Beer
    [39.0, 0.05, 1], # Expected: Whiskey
    [13.0, 1.3, 1]   # Expected: Wine
])

#normalising is not needed for decision trees
#test_data = (test_data - X_min) / (X_max - X_min)

tree = DecisionTree(max_depth=3)
tree.fit(X, y)
preds = tree.predict(test_data)

print("Predictions:", preds)
tree.print_tree()

Predictions: [1 2 0]
if Alcohol (ft 0) <= 8.5:
  Beer (1)
else:
  if Alcohol (ft 0) <= 25.75:
    Wine (0)
  else:
    Whiskey (2)
